# Crowd / 05 — Validation evaluation

Evaluate all candidates on 2023. Accuracy alone can reward majority prediction, so report macro F1 and balanced accuracy too.

Run every cell from top to bottom. Use the project `.venv` kernel in VS Code. No season field is used. Data and test scores are synthetic.

In [1]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml_service/workflows.py").exists())
sys.path.insert(0, str(ROOT))
from ml_service import workflows as w
print("Project:", ROOT)
print("Synthetic demonstration: real-world accuracy has not been measured.")

Project: /workspace/scratch/71e3a62599e6/YatraSense_Organic
Synthetic demonstration: real-world accuracy has not been measured.


In [2]:
print(inspect.getsource(w.evaluate_validation))

def evaluate_validation(task):
    df=pd.read_csv(DATA/f'{task}.csv'); va=df[df.dataset_split=='validation']
    features=CROWD_FEATURES if task=='crowd' else REC_FEATURES
    models=joblib.load(MODELS/f'{task}_candidates.joblib'); rows=[]
    for name,model in models.items():
        pred=model.predict(va[features]); result={'model':name,**metrics(va[TARGETS[task]],pred)}
        if task=='recommendation':
            prob=model.predict_proba(va[features])[:,1]
            result.update(ranking_metrics(va,prob));result['roc_auc']=float(roc_auc_score(va[TARGETS[task]],prob))
        rows.append(result)
    table=pd.DataFrame(rows).sort_values(['accuracy','macro_f1'],ascending=False)
    table.to_csv(REPORTS/f'{task}_validation.csv',index=False)
    return table



In [3]:
comparison = w.evaluate_validation("crowd")
display(comparison)

,model,accuracy,macro_f1,balanced_accuracy
1,RandomForest,0.770833,0.764700,0.751999
2,ExtraTrees,0.754167,0.751574,0.732952
3,GradientBoosting,0.745833,0.742319,0.726929
0,LogisticRegression,0.679167,0.655079,0.641860
4,MajorityBaseline,0.462500,0.210826,0.333333


The recommendation table also includes ROC AUC, NDCG@5 and Precision@5. NDCG excludes requests with no positive choice labels and reports the number used. A score is not a guarantee of usefulness for real customers.